In [11]:
from preprocessing import *

import sys
import numpy as np

sys.path.insert(0, '../..')

from project1.helpers import load_csv_data
from implementations import logistic_regression, reg_logistic_regression, ridge_regression

DATA_DIR = '../../project1/data'

In [7]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(DATA_DIR))
print('x_train:', x_train.shape)
print('x_test: ', x_test.shape)
print('y_train:', y_train.shape)
print('label counts:', {int(v): int(np.sum(y_train == v)) for v in np.unique(y_train)})


x_train: (328135, 321)
x_test:  (109379, 321)
y_train: (328135,)
label counts: {-1: 299160, 1: 28975}


In [14]:
audit_matrix(x_train, 'train')
audit_matrix(x_test, 'test')
for column in range(min(20, x_train.shape[1])):
    values, counts = np.unique(x_train[:, column], return_counts=True)
    order = np.argsort(counts)[-5:][::-1]
    print(column, list(zip(values[order].tolist(), counts[order].tolist())))

train (328135, 321)
  non-finite entries: 47175779
  constant columns: 6
  min/max: 0.0 2015023235.0
test (109379, 321)
  non-finite entries: 15724379
  constant columns: 5
  min/max: 0.0 2015023236.0
0 [(20.0, 17383), (31.0, 12959), (27.0, 12575), (53.0, 11997), (48.0, 10885)]
1 [(1.0, 32926), (10.0, 31335), (7.0, 30716), (4.0, 30272), (2.0, 27550)]
2 [(6022015.0, 1596), (6032015.0, 1587), (12032015.0, 1526), (3032015.0, 1518), (12092015.0, 1494)]
3 [(3.0, 31444), (11.0, 29486), (2.0, 29161), (1.0, 27937), (7.0, 27512)]
4 [(9.0, 13815), (8.0, 12911), (7.0, 12788), (14.0, 12744), (12.0, 12662)]
5 [(2015.0, 320011), (2016.0, 8124)]
6 [(1100.0, 278866), (1200.0, 49269)]
7 [(2015000426.0, 50), (2015000580.0, 49), (2015002274.0, 48), (2015001243.0, 48), (2015002455.0, 48)]
8 [(2015000426.0, 50), (2015000580.0, 49), (2015002274.0, 48), (2015001243.0, 48), (2015002455.0, 48)]
9 [(1.0, 188720), (nan, 139415)]
10 [(1.0, 188688), (nan, 139415), (2.0, 32)]
11 [(nan, 328103), (1.0, 32)]
12 [(1.0,

In [15]:
# 1. Replace survey-specific missing-value codes such as 9, 99, 999, etc. with NaN
x_train_clean = replace_survey_codes(x_train)
x_test_clean = replace_survey_codes(x_test)

# 2. Remove columns containing too many missing values
x_train_clean, x_test_clean = drop_columns_with_nan_fraction(
    x_train_clean,
    x_test_clean,
    max_fraction=0.50,
)

# 3. Remove constant columns, based on the training data
x_train_clean, x_test_clean = drop_constant_columns(
    x_train_clean,
    x_test_clean,
)

# 4. Fit imputation and standardization parameters using only training data
preprocessing_params = fit_preprocessor(x_train_clean)

# 5. Replace remaining NaNs with training medians and standardize both datasets
x_train_clean = transform_features(x_train_clean, preprocessing_params)
x_test_clean = transform_features(x_test_clean, preprocessing_params)

# 6. Add the intercept/bias column required by the regression models
x_train_clean = add_bias_column(x_train_clean)
x_test_clean = add_bias_column(x_test_clean)

print("Cleaned training data:", x_train_clean.shape)
print("Cleaned test data:", x_test_clean.shape)
print("Remaining NaNs in training data:", np.isnan(x_train_clean).sum())
print("Remaining NaNs in test data:", np.isnan(x_test_clean).sum())

Cleaned training data: (328135, 173)
Cleaned test data: (109379, 173)
Remaining NaNs in training data: 0
Remaining NaNs in test data: 0
